# 線性回歸

你是連鎖餐廳的執行長，要決定哪些城市值得開新店。第一部只用城市人口預測每月獲利。第二部再加上當地收入，並看特徵尺度不同時梯度下降會怎樣。

程式已經寫好，用來讓你看懂模型在做什麼。請 agent 改的格子，標題會寫「實驗」。下面這種格子只有你能寫，agent 不要填、不要改寫：


## 1 只用人口預測獲利

品牌已經在一些城市有店，手上有各城市人口，以及該店每月獲利（負值是虧損）。人口的單位是萬人，獲利的單位是萬美元。第一筆人口約 `4.62`，也就是約 46,200 人；獲利約 `2.82`，也就是每月約 28,200 美元。

先跑下一格，看資料長什麼樣子。


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

def find_data_dir() -> Path:
    return Path(__vsc_ipynb_file__).resolve().parent / "data"

DATA_DIR = find_data_dir()
rows = np.loadtxt(DATA_DIR / "ex1data1.csv", delimiter=",")
x_train = rows[:, 0]
y_train = rows[:, 1]
print("筆數 m =", len(x_train))
print("人口前 5 筆（萬人）", x_train[:5])
print("獲利前 5 筆（萬美元）", y_train[:5])

plt.scatter(x_train, y_train, marker="x", c="r")
plt.title("Profit vs. city population")
plt.xlabel("Population (10,000s)")
plt.ylabel("Profit ($10,000s)")
plt.show()


<div style="background:#fff6e0;border-left:6px solid #e09b00;padding:12px 16px;border-radius:4px;color:#1a1a1a;">

<h3 style="margin-top:0;color:#1a1a1a;">1 你的觀察</h3>
<p style="color:#1a1a1a;">看完資料點後再回答。</p>
<ol style="color:#1a1a1a;list-style:decimal;padding-left:1.4em;">
<li style="color:#1a1a1a;">人口變多時，獲利看起來怎麼走？</li>
<li style="color:#1a1a1a;">有沒有哪幾點明顯離群？</li>
</ol>

</div>


## 1.1 模型是一條線

單變量時，預測寫成

$$\hat{y} = wx + b$$

`w` 是斜率，人口每多一萬人，獲利預測要加多少。`b` 是人口為 0 時的基準，這份資料裡沒有這種城市，它只是把線搬上搬下。

下一格畫三條線，`b` 先固定為 0。這是實驗格，可以請 agent 改 `w` 再跑一次。


In [ ]:
# 1.1 實驗：拉滑桿，看一條線怎麼斜、怎麼上下移
import ipywidgets as widgets

x_line = np.linspace(x_train.min(), x_train.max(), 50)

def show_line(w, b):
    plt.figure()
    plt.scatter(x_train, y_train, marker="x", c="r", label="cities")
    plt.plot(x_line, w * x_line + b, label=f"w={w:.2f}, b={b:.2f}")
    plt.xlabel("Population (10,000s)")
    plt.ylabel("Profit ($10,000s)")
    plt.legend()
    plt.show()

widgets.interact(
    show_line,
    w=widgets.FloatSlider(value=1.0, min=-1.0, max=2.0, step=0.05, description="w"),
    b=widgets.FloatSlider(value=0.0, min=-10.0, max=10.0, step=0.5, description="b"),
)


## 1.2 Cost：這組 w、b 錯多少

每一個城市都有誤差：預測減掉實際獲利。Cost 是全部城市的平方誤差，再平均，前面乘 `1/(2m)`。平方讓正的誤差和負的誤差不會互相抵消。`1/2` 是為了後面微分時比較乾淨，不改變「哪一組 w、b 比較好」。

$$J(w,b) = \frac{1}{2m} \sum_{i=0}^{m-1} (wx^{(i)} + b - y^{(i)})^2$$

函式先給你看，不必重寫。


In [ ]:
def compute_cost(x, y, w, b):
    m = x.shape[0]
    total = 0.0
    for i in range(m):
        prediction = w * x[i] + b
        total += (prediction - y[i]) ** 2
    return total / (2 * m)

print(f"w=2, b=1 的 Cost = {compute_cost(x_train, y_train, 2, 1):.3f}")
print("預期結果是 62.639")


<div style="background:#fff6e0;border-left:6px solid #e09b00;padding:12px 16px;border-radius:4px;color:#1a1a1a;">

<h3 style="margin-top:0;color:#1a1a1a;">1.2 你的觀察</h3>
<ol style="color:#1a1a1a;list-style:decimal;padding-left:1.4em;">
<li style="color:#1a1a1a;">誤差直接相加變成 0，平方之後不是。若 Cost 用相加而不平方，會發生什麼事？</li>
</ol>

</div>


In [ ]:
# 1.2 實驗：拿掉平方會怎樣。三個點，誤差是 +1、0、-1。
x_tiny = np.array([1.0, 2.0, 3.0])
y_tiny = np.array([1.0, 2.0, 3.0])
w_try, b_try = 0.0, 2.0
errors = w_try * x_tiny + b_try - y_tiny
print("誤差", errors)
print("直接相加", errors.sum())
print("平方後相加", np.square(errors).sum())


## 1.3 這條曲線就是碗

誤差有平方，所以 `w` 太大或太小時 Cost 都會往上衝。下一格把 `b` 固定為 0，`w` 從 `-2.5` 掃到 `4`，畫出來的就是一隻開口向上的碗。圖上沒有 `b` 這一軸，碗還是在：最低的那一點，是 `b` 按在 0 時比較好的斜率。這些城市不在同一條直線上，所以碗底也不會是 0。


In [ ]:
w_grid = np.linspace(-2.5, 4, 80)
costs = [compute_cost(x_train, y_train, w, 0.0) for w in w_grid]
plt.plot(w_grid, costs)
plt.xlabel("w")
plt.ylabel("Cost J(w, b=0)")
plt.title("Cost bowl, one slice")
plt.show()


## 1.4 梯度下降：沿著碗往下走

訓練不是把公式解完，是重複做同一件事：看現在的 `w`、`b` 往哪邊改，Cost 會下降，就往那邊走一小步。步長是學習率 `alpha`。

$$\begin{align*} \text{重複}&\text{ 直到收斂:} \; \lbrace \newline
\;  w &= w -  \alpha \frac{\partial J(w,b)}{\partial w} \tag{3}  \; \newline
 b &= b -  \alpha \frac{\partial J(w,b)}{\partial b}  \newline \rbrace
\end{align*}$$

`w`、`b` 要同時更新：先把兩個偏微分都算完，再一起改。

單變量的梯度是：

$$
\begin{align}
\frac{\partial J(w,b)}{\partial w}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) - y^{(i)})x^{(i)} \tag{4}\\
  \frac{\partial J(w,b)}{\partial b}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) - y^{(i)}) \tag{5}\\
\end{align}
$$

下面的梯度與迴圈先寫好。跑出來的數字應對上預期結果。這一節不是叫你重寫函式。


In [ ]:
import copy
import math

def compute_gradient(x, y, w, b):
    m = x.shape[0]
    dj_dw = 0.0
    dj_db = 0.0
    for i in range(m):
        error = w * x[i] + b - y[i]
        dj_dw += error * x[i]
        dj_db += error
    return dj_dw / m, dj_db / m

def gradient_descent(x, y, w_in, b_in, alpha, num_iters):
    w = copy.deepcopy(w_in)
    b = b_in
    history = []
    for i in range(num_iters):
        dj_dw, dj_db = compute_gradient(x, y, w, b)
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
        history.append(compute_cost(x, y, w, b))
        if i % math.ceil(num_iters / 10) == 0:
            print(f"Epoch {i:4d}: Cost {history[-1]:8.2f}")
    return w, b, history

dj_dw, dj_db = compute_gradient(x_train, y_train, 0.0, 0.0)
print("w=0, b=0 的梯度", dj_dw, dj_db)
print("預期結果約 -52.755 與 -5.756")

w, b, history = gradient_descent(x_train, y_train, 0.0, 0.0, alpha=0.01, num_iters=1500)
print("學到的 w, b =", w, b)
print("預期結果約 w=1.060, b=-2.443")

plt.plot(history)
plt.xlabel("epoch")
plt.ylabel("Cost")
plt.title("Cost should fall, then flatten")
plt.show()

x_line = np.linspace(x_train.min(), x_train.max(), 50)
plt.scatter(x_train, y_train, marker="x", c="r")
plt.plot(x_line, w * x_line + b, c="b")
plt.xlabel("Population (10,000s)")
plt.ylabel("Profit ($10,000s)")
plt.title("Fitted line")
plt.show()

for population in (3.5, 7.0):
    profit = (population * w + b) * 10000
    print(f"人口 {population * 10000:.0f} 人，預測每月獲利 ${profit:,.0f}")


<div style="background:#fff6e0;border-left:6px solid #e09b00;padding:12px 16px;border-radius:4px;color:#1a1a1a;">

<h3 style="margin-top:0;color:#1a1a1a;">1.4 你的觀察</h3>
<ol style="color:#1a1a1a;list-style:decimal;padding-left:1.4em;">
<li style="color:#1a1a1a;">Cost 對 epoch 那張圖，最後有沒有變平？變平的時候，<code style="color:#1a1a1a;">w</code> 和 <code style="color:#1a1a1a;">b</code> 是已經靠近碗底，還是還在大步往下走？</li>
<li style="color:#1a1a1a;">人口對獲利那張圖，藍線有沒有穿過大部分紅點？人口很少和人口很多的城市，哪一邊離線比較遠？</li>
<li style="color:#1a1a1a;">人口 35,000 和 70,000 各預測了多少獲利。若真的要開店，你會比較相信哪一個？</li>
</ol>

</div>


## 2 人口加上收入

同一間公司，這次每個城市有兩個數：人口，以及當地收入的指標。預測仍是每月獲利。

模型改成向量。`X` 的每一列是一個城市，`w` 有兩個數，一個對人口、一個對收入。

$$\hat{y} = Xw + b$$

`np.dot(X, w)` 就是每個城市把兩個特徵各乘上自己的 `w` 再相加。這是測驗 11 用的寫法，迴圈版和向量版算的是同一件事。


In [ ]:
multi = np.loadtxt(DATA_DIR / "ex1data2.csv", delimiter=",")
X_train = multi[:, :2]
y_multi = multi[:, 2]
names = ["population (10,000s)", "income proxy ($10,000s)"]
print("形狀", X_train.shape, "特徵標準差", X_train.std(axis=0))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for j in range(2):
    axes[j].scatter(X_train[:, j], y_multi, marker="x", c="r")
    axes[j].set_xlabel(names[j])
    axes[j].set_ylabel("profit ($10,000s)")
plt.tight_layout()
plt.show()


## 2.1 尺度不同時，同一步長會踩太大

人口的變動範圍比收入大。梯度下降若用同一個 `alpha`，大尺度的特徵會把 `w` 推得很遠，Cost 可能先爆再亂跳。

z-score 是每個特徵先減自己的平均、再除以自己的標準差，讓兩個特徵都落在差不多的尺度。預測新城市時要用訓練時的平均和標準差，不能重新算一組。

下一格用同一個 `alpha=0.3` 跑原始資料和縮放後的資料。這是實驗格。


In [ ]:
def zscore_normalize_features(X):
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    return (X - mu) / sigma, mu, sigma

def compute_cost_multi(X, y, w, b):
    m = X.shape[0]
    prediction = np.dot(X, w) + b
    return np.sum((prediction - y) ** 2) / (2 * m)

def compute_gradient_multi(X, y, w, b):
    m = X.shape[0]
    errors = np.dot(X, w) + b - y
    dj_dw = np.dot(X.T, errors) / m
    dj_db = np.sum(errors) / m
    return dj_dw, dj_db

def gradient_descent_multi(X, y, alpha, num_iters):
    w = np.zeros(X.shape[1])
    b = 0.0
    history = []
    for _ in range(num_iters):
        dj_dw, dj_db = compute_gradient_multi(X, y, w, b)
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
        history.append(compute_cost_multi(X, y, w, b))
    return w, b, history

X_norm, mu, sigma = zscore_normalize_features(X_train)
zeros_w = np.zeros(X_norm.shape[1])
print(f"縮放後、w=0 的 Cost = {compute_cost_multi(X_norm, y_multi, zeros_w, 0.0):.3f}")
print("預期結果是 120.388")

dj_dw, dj_db = compute_gradient_multi(X_norm, y_multi, zeros_w, 0.0)
print("梯度 dj_dw", dj_dw, "dj_db", round(float(dj_db), 3))
print("預期結果約 dj_dw=[-5.131, -4.889], dj_db=-14.111")

# 2.1 實驗：同一個 alpha。原始尺度的 Cost 會衝上去，縮放後才往下。
alpha = 0.3
_, _, raw_hist = gradient_descent_multi(X_train, y_multi, alpha, 30)
_, _, norm_hist = gradient_descent_multi(X_norm, y_multi, alpha, 30)
print("原始資料前 5 次 Cost", [round(float(v), 2) for v in raw_hist[:5]])
print("縮放後前 5 次 Cost", [round(float(v), 2) for v in norm_hist[:5]])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(raw_hist)
axes[0].set_title("raw features, alpha=0.3")
axes[1].plot(norm_hist)
axes[1].set_title("z-score, alpha=0.3")
for ax in axes:
    ax.set_xlabel("epoch")
    ax.set_ylabel("Cost")
plt.tight_layout()
plt.show()


In [ ]:
w_multi, b_multi, history = gradient_descent_multi(X_norm, y_multi, alpha=0.005, num_iters=1000)
print("學到的 w, b =", w_multi, b_multi)
print("預期結果約 w=[4.052, 3.722], b=14.017")

y_pred = np.dot(X_norm, w_multi) + b_multi
plt.scatter(y_multi, y_pred, marker="x", c="r")
lims = [min(y_multi.min(), y_pred.min()), max(y_multi.max(), y_pred.max())]
plt.plot(lims, lims, "k--", label="perfect")
plt.xlabel("Actual profit ($10,000s)")
plt.ylabel("Predicted profit ($10,000s)")
plt.legend()
plt.show()

# 有人提議在一座還沒開過店的城市試點。那裡約 16 萬人，當地收入指標約 7 萬美元。
# 模型是用縮放後的人口、收入學的，不能把 16 和 7 直接乘上 w。
# 先用訓練城市的 mu、sigma 做同樣的縮放，再預測每月獲利。
new_city = np.array([16.0, 7.0])  # 人口（萬人）、收入指標（萬美元）
new_norm = (new_city - mu) / sigma
print("這座新城的預測每月獲利（萬美元）", float(np.dot(new_norm, w_multi) + b_multi))


## 2.2 只用人口，和再加上收入

第一部的城市和第二部不是同一批，兩邊的 Cost 不能拿來比。要比單變量和多變量，得用同一批城市、同一個學習率、同樣的步數。

下一格在縮放後的資料上跑兩次：一次只留人口，一次人口加收入。看最後的 Cost，以及同一座新城的預測差多少。


In [ ]:
X_pop = X_norm[:, :1]
w_pop, b_pop, _ = gradient_descent_multi(X_pop, y_multi, alpha=0.005, num_iters=1000)
cost_pop = compute_cost_multi(X_pop, y_multi, w_pop, b_pop)
cost_both = compute_cost_multi(X_norm, y_multi, w_multi, b_multi)
print("只用人口的 Cost", round(float(cost_pop), 3))
print("人口加收入的 Cost", round(float(cost_both), 3))

new_pop = (np.array([16.0]) - mu[0]) / sigma[0]
print("只用人口，這座新城", float(np.dot(new_pop, w_pop) + b_pop))
print("人口加收入，這座新城", float(np.dot(new_norm, w_multi) + b_multi))


In [ ]:
# 同一組縮放後的資料，用 sklearn 對一下係數。係數應接近上面的 w、b。
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_norm, y_multi)
print("sklearn w, b =", model.coef_, model.intercept_)
print("梯度下降 w, b =", w_multi, b_multi)
print("係數差", model.coef_ - w_multi, "截距差", model.intercept_ - b_multi)
print("兩邊是同一個直線模型。差一點通常是梯度下降的步數還沒走完。")


<div style="background:#fff6e0;border-left:6px solid #e09b00;padding:12px 16px;border-radius:4px;color:#1a1a1a;">

<h3 style="margin-top:0;color:#1a1a1a;">2 你的觀察</h3>
<ol style="color:#1a1a1a;list-style:decimal;padding-left:1.4em;">
<li style="color:#1a1a1a;">兩個特徵裡，你覺得哪個和獲利的關係比較清楚：</li>
<li style="color:#1a1a1a;">不縮放時 Cost 發生了什麼。縮放之後呢：</li>
<li style="color:#1a1a1a;">sklearn 的係數和自己跑的梯度下降差多少。差很大的話，你會先懷疑哪裡：</li>
<li style="color:#1a1a1a;">同一批城市，只用人口和再加上收入，Cost 哪一個比較小？同一座新城的兩個預測差多少，多出來的收入有沒有改你的判斷：</li>
</ol>

</div>


實驗本到這裡。接下來打開同資料夾的 `brief.md`，在 `cabbage_pricing.ipynb` 裡做估價。
